# Prova de Conceito: Fusão Precoce via API da OpenAI com Structured Output

## Configurando client

In [11]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

# Definir modelo que será utilizado no método (quanto mais potente, maior acurárica, porém maior o custo)
MODEL = "gpt-5.5"

# Carrega a key de uma variável de ambiente
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

## Caso 1: Documento estruturado (CNH)

In [12]:
import base64
import json

# --- 1. Carregar e converter a imagem para base64 ---
caminho_imagem = "documents/cnh.jpeg"

with open(caminho_imagem, "rb") as f:
    base64_image = base64.b64encode(f.read()).decode("utf-8")

image_url = f"data:image/jpeg;base64,{base64_image}"

# --- 2. Schema da CNH - Utilizado para gerar o JSON esperado (structured output) ---
schema_cnh = {
    "type": "object",
    "properties": {
        "nome": {"type": "string"},
        "cpf": {"type": "string"},
        "data_nascimento": {"type": "string"},
        "data_emissao": {"type": "string"},
        "filiacao_pai": {"type": "string"},
        "filiacao_mae": {"type": "string"}
    },
    "required": ["nome", "cpf", "data_nascimento", "data_emissao", "filiacao_pai", "filiacao_mae"],
    "additionalProperties": False
}

# --- 3. Chamada à API ---
response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Extraia os dados dessa CNH."},
                {"type": "input_image", "image_url": image_url}
            ]
        }
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "extracao_cnh",
            "schema": schema_cnh,
            "strict": True
        }
    }
)
# --- 4. Resultado ---
dados_extraidos = json.loads(response.output_text)
print("Dados extraídos:", json.dumps(dados_extraidos, indent=2, ensure_ascii=False))

Dados extraídos: {
  "nome": "CINDE DA SILVA",
  "cpf": "891.340.611-72",
  "data_nascimento": "02/08/2017",
  "data_emissao": "22/10/2017",
  "filiacao_pai": "José da Silva",
  "filiacao_mae": "Maria da Silva"
}


## Caso 2: Documento extenso (The Claude 3 Model Family)

In [ ]:
"""
    Por restrição de custo, esta célula (processamento de todas as páginas do documento)
    não foi executada nesta POC.
"""

import base64
import json
from pdf2image import convert_from_path

# --- 1. Converter TODAS as páginas do PDF em imagens ---
caminho_pdf = "documents/paper.pdf"
paginas = convert_from_path(caminho_pdf, dpi=200)

caminhos_imagens = []
for i, pagina in enumerate(paginas):
    caminho = f"documents/paper_pagina_{i+1}.png"
    pagina.save(caminho, "PNG")
    caminhos_imagens.append(caminho)

def imagem_para_base64_url(caminho):
    with open(caminho, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/png;base64,{b64}"

# --- 2. Montar a lista de conteúdo (1 input_image por página) ---
conteudo = [
    {"type": "input_text", "text": "Analise todas as páginas deste paper. Identifique o título do documento, as seções presentes, extraia as tabelas relevantes (mantendo organização/layout) e interprete os gráficos e imagens não-textuais encontrados."}
]

for caminho in caminhos_imagens:
    conteudo.append({"type": "input_image", "image_url": imagem_para_base64_url(caminho)})

# --- 3. Schema para o documento extenso ---
schema_paper = {
    "type": "object",
    "properties": {
        "titulo_documento": {"type": "string"},
        "secoes_identificadas": {"type": "array", "items": {"type": "string"}},
        "tabelas_extraidas": {
            "type": "array",
            "items": {"type": "string"},
            "description": "conteúdo de cada tabela relevante, em formato markdown"
        },
        "graficos_interpretados": {
            "type": "array",
            "items": {"type": "string"},
            "description": "descrição textual de cada gráfico/figura relevante encontrado"
        }
    },
    "required": ["titulo_documento", "secoes_identificadas", "tabelas_extraidas", "graficos_interpretados"],
    "additionalProperties": False
}

# --- 4. Chamada à API com todas as páginas ---
### Por motivo de custo elevado de chamada, optei por não rodar com a minha API
response = client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": conteudo}],
    text={
        "format": {
            "type": "json_schema",
            "name": "extracao_paper_completo",
            "schema": schema_paper,
            "strict": True
        }
    }
)

# --- 5. Resultado ---
dados_extraidos = json.loads(response.output_text)
print("Dados extraídos:", json.dumps(dados_extraidos, indent=2, ensure_ascii=False))

## Caso 3 - Documento com Layout Complexo (Fatura de energia)

In [9]:
import base64
import json

# --- 1. Carregar e converter a imagem para base64 ---
caminho_imagem = "documents/fatura.jpg"

with open(caminho_imagem, "rb") as f:
    base64_image = base64.b64encode(f.read()).decode("utf-8")

image_url = f"data:image/jpeg;base64,{base64_image}"

# --- 2. Schema da fatura de energia ---
schema_fatura = {
    "type": "object",
    "properties": {
        "nome_titular": {"type": "string"},
        "numero_instalacao": {"type": "string"},
        "mes_referencia": {"type": "string"},
        "valor_total": {"type": "number"},
        "data_vencimento": {"type": "string"},
        "consumo_kwh": {"type": "number"},
        "historico_consumo": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "mes": {"type": "string"},
                    "kwh": {"type": "number"}
                },
                "required": ["mes", "kwh"],
                "additionalProperties": False
            }
        }
    },
    "required": ["nome_titular", "numero_instalacao", "mes_referencia", "valor_total", "data_vencimento", "consumo_kwh", "historico_consumo"],
    "additionalProperties": False
}

# --- 3. Chamada à API ---
response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Extraia os dados dessa fatura de energia, incluindo o histórico de consumo mostrado no gráfico, se houver."},
                {"type": "input_image", "image_url": image_url}
            ]
        }
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "extracao_fatura",
            "schema": schema_fatura,
            "strict": True
        }
    }
)

# --- 4. Resultado ---
dados_extraidos = json.loads(response.output_text)
print("Dados extraídos:", json.dumps(dados_extraidos, indent=2, ensure_ascii=False))

Dados extraídos: {
  "nome_titular": "MARIA JOSÉ DA SILVA",
  "numero_instalacao": "1234567890",
  "mes_referencia": "06/2014",
  "valor_total": 63.72,
  "data_vencimento": "10/07/2014",
  "consumo_kwh": 160,
  "historico_consumo": [
    {
      "mes": "JUL/14",
      "kwh": 155
    },
    {
      "mes": "JUN/14",
      "kwh": 160
    },
    {
      "mes": "MAI/14",
      "kwh": 168
    },
    {
      "mes": "ABR/14",
      "kwh": 169
    },
    {
      "mes": "MAR/14",
      "kwh": 152
    },
    {
      "mes": "FEV/14",
      "kwh": 157
    },
    {
      "mes": "JAN/14",
      "kwh": 151
    },
    {
      "mes": "DEZ/13",
      "kwh": 159
    },
    {
      "mes": "NOV/13",
      "kwh": 168
    },
    {
      "mes": "OUT/13",
      "kwh": 174
    },
    {
      "mes": "SET/13",
      "kwh": 150
    },
    {
      "mes": "AGO/13",
      "kwh": 165
    },
    {
      "mes": "JUL/13",
      "kwh": 156
    }
  ]
}
